# RNN & LSTM Practice — Stock Market Price Prediction

**Task:** (1) find a relevant problem, (2) find relevant data, (3) demonstrate an RNN / LSTM in Google Colab.

## 1. The problem
Stock indices are a classic **time-series** problem: today's movement depends on the sequence of previous days, which is exactly what recurrent networks are designed for.

> **Goal:** given the previous **60 trading days** of the NASDAQ Composite (`IXIC`), predict the **next trading day's close**.

## 2. The data
`Market.csv` — daily Open / High / Low / Close / Adj Close / Volume for **14 world stock indices** (1965–2021). We use one index at a time (default `IXIC`, ~12,700 days, no missing values). Change `INDEX` below to try another (e.g. `NYA`, `N225`, `GDAXI`, `HSI`).

## 3. Approach (and why)
| Decision | Reason |
|---|---|
| Predict **log-returns**, then convert back to price | NASDAQ went from ~55 to ~14,000. A network fed raw scaled prices can't output values outside the range it trained on, so it fails on the test period. Returns are stationary and on a stable scale. |
| **Chronological** train / val / test split (70/15/15) | Shuffling would leak the future into training. |
| Scaler fitted on **train only** | Avoids data leakage. |
| Compare against a **naive baseline** ("tomorrow = today") | Stock prices are very persistent, so almost any model looks great on a price chart. The baseline tells us if the network actually learned something. |
| LSTM **and** SimpleRNN | Shows the effect of LSTM gates on long sequences. |

In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__)

## Load the data
If `Market.csv` isn't in the Colab file panel, the cell below opens an upload dialog.

In [ ]:
CSV = "Market.csv"
if not os.path.exists(CSV):
    try:
        from google.colab import files
        print("Please upload Market.csv")
        uploaded = files.upload()
        CSV = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError("Put Market.csv next to this notebook.")

raw = pd.read_csv(CSV)
raw["Date"] = pd.to_datetime(raw["Date"], format="%m/%d/%Y")
print("Indices available:", sorted(raw["Index"].unique()))

INDEX = "IXIC"      # <- change to try another index
df = (raw[raw["Index"] == INDEX]
      .sort_values("Date").set_index("Date")[["Open", "High", "Low", "Close"]]
      .dropna())
df = df[df["Close"] > 0]
print(f"{INDEX}: {len(df):,} trading days, {df.index.min().date()} -> {df.index.max().date()}")
df.tail()

## Explore the data

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax[0].plot(df.index, df["Close"]); ax[0].set_yscale("log")
ax[0].set_title(f"{INDEX} close price (log scale)"); ax[0].grid(alpha=.3)
ret_plot = np.log(df["Close"]).diff()
ax[1].plot(df.index, ret_plot, lw=.5); ax[1].set_title("Daily log-return"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
print(ret_plot.describe())

The price trends upward exponentially (non-stationary), while the daily return hovers around zero with volatility clusters (1987, 2000, 2008, 2020). That is why we model **returns**.

## Build features & sliding windows
Each sample = the previous `WINDOW` days of 2 features → target = **next-day log-return**:
* `ret`   — log-return of Close
* `range` — intraday range `(High − Low) / Close` (a volatility proxy)

In [ ]:
WINDOW = 60

feat = pd.DataFrame(index=df.index)
feat["ret"]   = np.log(df["Close"]).diff()
feat["range"] = (df["High"] - df["Low"]) / df["Close"]
feat = feat.dropna()
close = df["Close"].loc[feat.index].values
F = feat.values.astype("float64")             # (T, 2)
ret = feat["ret"].values
n_feat = F.shape[1]

# sample k: window = rows [k, k+W)  ->  target row t = k+W
n = len(F) - WINDOW
n_train, n_val = int(n * 0.70), int(n * 0.15)
n_test = n - n_train - n_val

# scalers use TRAIN data only
scaler = StandardScaler().fit(F[: WINDOW + n_train - 1])
y_scaler = StandardScaler().fit(ret[WINDOW : WINDOW + n_train].reshape(-1, 1))
F_s = scaler.transform(F)

X = np.stack([F_s[k : k + WINDOW] for k in range(n)]).astype("float32")      # (n, W, 2)
y = y_scaler.transform(ret[WINDOW:].reshape(-1, 1)).astype("float32")        # (n, 1)

tgt_rows   = np.arange(WINDOW, len(F))
dates      = feat.index[tgt_rows]
prev_close = close[tgt_rows - 1]
true_close = close[tgt_rows]
true_ret   = ret[tgt_rows]

s1, s2 = n_train, n_train + n_val
X_tr, y_tr = X[:s1],   y[:s1]
X_va, y_va = X[s1:s2], y[s1:s2]
X_te, y_te = X[s2:],   y[s2:]
print(f"train {len(X_tr):,} | val {len(X_va):,} | test {len(X_te):,}   X shape: {X.shape}")
print("train :", dates[0].date(), "->", dates[s1-1].date())
print("val   :", dates[s1].date(), "->", dates[s2-1].date())
print("test  :", dates[s2].date(), "->", dates[-1].date())

## Build the models
**LSTM:** two stacked LSTM layers (64 → 32 units) with dropout, then a single Dense neuron predicting the next (scaled) return.
**SimpleRNN:** same idea with a plain recurrent layer, for comparison.

In [ ]:
def build_lstm():
    return keras.Sequential([
        layers.Input(shape=(WINDOW, n_feat)),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(32),
        layers.Dropout(0.2),
        layers.Dense(1),
    ], name="lstm")

def build_rnn():
    return keras.Sequential([
        layers.Input(shape=(WINDOW, n_feat)),
        layers.SimpleRNN(64, return_sequences=True),
        layers.Dropout(0.2),
        layers.SimpleRNN(32),
        layers.Dropout(0.2),
        layers.Dense(1),
    ], name="simple_rnn")

def train(model, epochs=40):
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)]
    return model.fit(X_tr, y_tr, validation_data=(X_va, y_va),
                     epochs=epochs, batch_size=64, callbacks=cb, verbose=2)

lstm = build_lstm(); lstm.summary()

## Train

In [ ]:
hist_lstm = train(lstm)
rnn = build_rnn()
hist_rnn = train(rnn)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, h, name in zip(ax, [hist_lstm, hist_rnn], ["LSTM", "SimpleRNN"]):
    a.plot(h.history["loss"], label="train"); a.plot(h.history["val_loss"], label="validation")
    a.set_title(f"{name} loss (MSE)"); a.set_xlabel("epoch"); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Evaluate on the unseen test period
Predictions are **one-step-ahead**: the model predicts tomorrow's return, which we turn into a price with `price_t = close_(t-1) × exp(return_t)`.

Metrics: **RMSE / MAE / MAPE** on price, and **directional accuracy** (did we call up vs. down correctly?). The naive baseline predicts "no change"; for direction we use "always up".

In [ ]:
def to_price(pred_scaled):
    r = y_scaler.inverse_transform(np.asarray(pred_scaled).reshape(-1, 1)).ravel()
    return r, prev_close[s2:] * np.exp(r)

actual_price = true_close[s2:]
actual_ret   = true_ret[s2:]

def metrics(name, price_pred, ret_pred):
    err = price_pred - actual_price
    return {"model": name,
            "RMSE": np.sqrt(np.mean(err**2)),
            "MAE":  np.mean(np.abs(err)),
            "MAPE %": np.mean(np.abs(err / actual_price)) * 100,
            "Direction acc %": np.mean(np.sign(ret_pred) == np.sign(actual_ret)) * 100}

r_lstm, p_lstm = to_price(lstm.predict(X_te, verbose=0))
r_rnn,  p_rnn  = to_price(rnn.predict(X_te, verbose=0))
p_naive = prev_close[s2:]

rows = [metrics("LSTM", p_lstm, r_lstm),
        metrics("SimpleRNN", p_rnn, r_rnn),
        metrics("Naive (yesterday's close)", p_naive, np.zeros_like(actual_ret))]
rows[-1]["Direction acc %"] = np.mean(actual_ret > 0) * 100      # 'always up' baseline
results = pd.DataFrame(rows).set_index("model").round(3)
results

In [ ]:
N = 250   # last ~1 trading year of the test set
d = dates[s2:][-N:]
fig, ax = plt.subplots(2, 1, figsize=(12, 8))
ax[0].plot(d, actual_price[-N:], label="Actual", lw=2, color="black")
ax[0].plot(d, p_lstm[-N:], label="LSTM", lw=1.3)
ax[0].plot(d, p_naive[-N:], label="Naive", lw=1, ls="--", alpha=.7)
ax[0].set_title(f"{INDEX}: actual vs predicted close (last {N} test days)"); ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].plot(d, actual_ret[-N:], label="Actual return", color="black", lw=.8)
ax[1].plot(d, r_lstm[-N:], label="LSTM predicted return", lw=.8)
ax[1].set_title("Daily return: actual vs predicted"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Forecast the next trading day
Uses the most recent 60 days in the file. **For learning only — not financial advice.**

In [ ]:
last_window = scaler.transform(F[-WINDOW:])[None].astype("float32")
r_next = y_scaler.inverse_transform(lstm.predict(last_window, verbose=0))[0, 0]
last_close = close[-1]
print(f"Last date in data : {feat.index[-1].date()}  close = {last_close:,.2f}")
print(f"Predicted return  : {r_next*100:+.3f}%")
print(f"Predicted next close = {last_close*np.exp(r_next):,.2f}")

## Conclusion
* **RNN vs LSTM:** compare the two rows in the results table and their loss curves. LSTM gates (forget / input / output) help keep information across a 60-step sequence, while a plain RNN suffers from vanishing gradients on long windows.
* **Price error vs. baseline:** on a *price* chart, the LSTM line hugs the actual line, but so does the naive "tomorrow = today" line, because prices change only ~1% per day. If the LSTM's RMSE is about the same as the baseline's, that is the **expected, honest result** — it did not "beat the market".
* **Direction accuracy:** compare against the *always-up* baseline in the table (markets rose over the test period, so "always up" already scores above 50 %). Expect the networks to land near 50–55 %; if they don't beat the baseline, that's a real finding, not a bug. Daily returns are mostly noise, which is why stock prediction is hard (efficient-market hypothesis).
* **Improvements to try:** more features (volume, moving averages, RSI, other indices), a different `WINDOW`, `Bidirectional`/`GRU` layers, predicting several days ahead, or walk-forward validation.

> Disclaimer: educational demo. Do not use it for real trading decisions.